In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
 

In [4]:
df = pd.read_csv(r"C:\Users\DELL\Pictures\Documents\CodeBind-projects\WeatherPrediction\weatherHistory.csv")
 
print("Shape of dataset:", df.shape)
print(df.head())
print(df.info())

Shape of dataset: (96453, 12)
                  Formatted Date        Summary Precip Type  Temperature (C)  \
0  2006-04-01 00:00:00.000 +0200  Partly Cloudy        rain         9.472222   
1  2006-04-01 01:00:00.000 +0200  Partly Cloudy        rain         9.355556   
2  2006-04-01 02:00:00.000 +0200  Mostly Cloudy        rain         9.377778   
3  2006-04-01 03:00:00.000 +0200  Partly Cloudy        rain         8.288889   
4  2006-04-01 04:00:00.000 +0200  Mostly Cloudy        rain         8.755556   

   Apparent Temperature (C)  Humidity  Wind Speed (km/h)  \
0                  7.388889      0.89            14.1197   
1                  7.227778      0.86            14.2646   
2                  9.377778      0.89             3.9284   
3                  5.944444      0.83            14.1036   
4                  6.977778      0.83            11.0446   

   Wind Bearing (degrees)  Visibility (km)  Loud Cover  Pressure (millibars)  \
0                   251.0          15.8263      

In [5]:
print("\nMissing values per column:\n", df.isnull().sum())


Missing values per column:
 Formatted Date                0
Summary                       0
Precip Type                 517
Temperature (C)               0
Apparent Temperature (C)      0
Humidity                      0
Wind Speed (km/h)             0
Wind Bearing (degrees)        0
Visibility (km)               0
Loud Cover                    0
Pressure (millibars)          0
Daily Summary                 0
dtype: int64


In [6]:
print("\nStatistical summary:\n", df.describe())


Statistical summary:
        Temperature (C)  Apparent Temperature (C)      Humidity  \
count     96453.000000              96453.000000  96453.000000   
mean         11.932678                 10.855029      0.734899   
std           9.551546                 10.696847      0.195473   
min         -21.822222                -27.716667      0.000000   
25%           4.688889                  2.311111      0.600000   
50%          12.000000                 12.000000      0.780000   
75%          18.838889                 18.838889      0.890000   
max          39.905556                 39.344444      1.000000   

       Wind Speed (km/h)  Wind Bearing (degrees)  Visibility (km)  Loud Cover  \
count       96453.000000            96453.000000     96453.000000     96453.0   
mean           10.810640              187.509232        10.347325         0.0   
std             6.913571              107.383428         4.192123         0.0   
min             0.000000                0.000000         0

In [7]:
df = df.drop(columns=['Formatted Date', 'Summary', 'Daily Summary', 'Loud Cover'])

In [8]:
df['Precip Type'] = df['Precip Type'].fillna(df['Precip Type'].mode()[0])

In [9]:
df = pd.get_dummies(df, columns=['Precip Type'], drop_first=True)

In [10]:
# 3.6 Correlation heatmap - see which features relate to our target
plt.figure(figsize=(10, 6))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.savefig('Correlation_Heatmap.png')
plt.close()

In [11]:
# 3.7 Distribution of target variable
plt.figure(figsize=(8, 5))
sns.histplot(df['Apparent Temperature (C)'], kde=True)
plt.title('Distribution of Apparent Temperature')
plt.savefig('Target_Distribution.png')
plt.close()

In [12]:
# 3.8 Scatter plot: Humidity vs Apparent Temperature
plt.figure(figsize=(8, 5))
sns.scatterplot(x='Humidity', y='Apparent Temperature (C)', data=df, alpha=0.3)
plt.title('Humidity vs Apparent Temperature')
plt.savefig('Humidity_Vs_Temperature.png')
plt.close()


In [13]:
target = 'Apparent Temperature (C)'
 
# Features: everything except the target
features = [col for col in df.columns if col != target]
 
X = df[features]
y = df[target]
 
print("\nFeatures used:", features)


Features used: ['Temperature (C)', 'Humidity', 'Wind Speed (km/h)', 'Wind Bearing (degrees)', 'Visibility (km)', 'Pressure (millibars)', 'Precip Type_snow']


In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
 
print("\nTraining samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])
 


Training samples: 77162
Testing samples: 19291


In [15]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
 

In [16]:
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
lr_preds = lr_model.predict(X_test_scaled)
 

In [17]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)   # tree models don't need scaling
rf_preds = rf_model.predict(X_test)
print(rf_preds)
 

[28.85422222  7.98888889 14.4        ... 10.71672222 24.889
 18.86111111]


In [18]:
def evaluate(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    print(f"\n--- {name} ---")
    print(f"MAE  : {mae:.3f}")
    print(f"MSE  : {mse:.3f}")
    print(f"RMSE : {rmse:.3f}")
    print(f"R2   : {r2:.3f}")
    return mae, mse, rmse, r2
 
evaluate("Linear Regression", y_test, lr_preds)
evaluate("Random Forest", y_test, rf_preds)


--- Linear Regression ---
MAE  : 0.844
MSE  : 1.146
RMSE : 1.070
R2   : 0.990

--- Random Forest ---
MAE  : 0.014
MSE  : 0.003
RMSE : 0.055
R2   : 1.000


(0.014175332192911913,
 0.0030027953785779888,
 np.float64(0.054797768007264575),
 0.9999740464394473)

In [19]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, rf_preds, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Apparent Temperature')
plt.ylabel('Predicted Apparent Temperature')
plt.title('Random Forest: Actual vs Predicted')
plt.savefig('Actual_vs_Predicted.png')
plt.close()
 

In [20]:
sample = X_test.iloc[0:1]
actual_value = y_test.iloc[0]
predicted_value = rf_model.predict(sample)[0]
 
print("\nSample Prediction:")
print("Actual Apparent Temperature   :", actual_value)
print("Predicted Apparent Temperature:", predicted_value)


Sample Prediction:
Actual Apparent Temperature   : 28.827777777777776
Predicted Apparent Temperature: 28.854222222222212
